# FixaTons Tutorial

This notebook is the fastest path from a fresh install to a small, reproducible eye-tracking analysis.

What you will do:
- inspect the bundled datasets
- load one stimulus and its human-derived maps
- visualize fixation and scanpath data
- compute a few saliency and scanpath metrics
- validate the dataset artifacts shipped with the repository

The notebook uses the small `SIENA12` dataset first, because it is compact and easy to understand.

## 1. Import the package

The direct top-level API is the recommended interface.
If you installed from source inside this repository, the bundled datasets under `Dataset/` are discovered automatically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import FixaTons as ft
from FixaTons import dataset_utils

## 2. Inspect the collection

Start by checking which datasets are available and where FixaTons is reading them from.

In [ ]:
print("Collection path:", ft.COLLECTION_PATH)
print("Available datasets:", ft.info.datasets())

## 3. Pick one dataset and stimulus

We will use one stimulus from `SIENA12` throughout the tutorial. This keeps the examples concrete and makes the outputs easy to interpret.

In [ ]:
dataset_name = "SIENA12"
stimulus_name = "sunset.jpg"

subjects = ft.info.subjects(dataset_name, stimulus_name)
print("Stimulus:", stimulus_name)
print("Subjects watching it:", len(subjects))
print("First 5 subject IDs:", subjects[:5])

## 4. Load the main data objects

A single stimulus can be represented in several complementary ways:
- `stimulus`: the original image
- `fixation_map`: a binary map of fixated locations
- `saliency_map`: a smoothed map generated from human fixations
- `scanpath`: one subject's time-ordered fixation sequence

In [ ]:
stimulus = ft.stimulus(dataset_name, stimulus_name)
fixation_map = ft.fixation_map(dataset_name, stimulus_name)
saliency_map = ft.saliency_map(dataset_name, stimulus_name)
scanpath = ft.scanpath(dataset_name, stimulus_name, subject=subjects[0])

print("stimulus shape:", stimulus.shape)
print("fixation_map shape:", fixation_map.shape, "unique values:", np.unique(fixation_map))
print("saliency_map shape:", saliency_map.shape, "value range:", (float(saliency_map.min()), float(saliency_map.max())))
print("scanpath shape:", scanpath.shape)
print("first 3 fixations:\n", scanpath[:3])

## 5. Visualize what the data means

The first row shows the original stimulus, the binary fixation map, and the human saliency map.
The second row overlays one subject's scanpath on top of the original image.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(stimulus[:, :, ::-1])
axes[0, 0].set_title("Stimulus")
axes[0, 1].imshow(fixation_map, cmap="gray")
axes[0, 1].set_title("Fixation Map")
axes[0, 2].imshow(saliency_map, cmap="magma")
axes[0, 2].set_title("Saliency Map")

axes[1, 0].imshow(stimulus[:, :, ::-1])
axes[1, 0].scatter(scanpath[:, 0], scanpath[:, 1], c=np.arange(len(scanpath)), cmap="viridis", s=20)
axes[1, 0].plot(scanpath[:, 0], scanpath[:, 1], color="white", linewidth=1)
axes[1, 0].set_title("One Subject Scanpath")

axes[1, 1].imshow(stimulus[:, :, ::-1])
for i, fixation in enumerate(scanpath):
    axes[1, 1].text(fixation[0] + 3, fixation[1] + 3, str(i + 1), color="white", fontsize=10)
axes[1, 1].set_title("One Subject Fixations")

axes[1, 2].axis("off")
axes[1, 2].text(
    0,
    0.9,
    "Scanpath row format:\n[x_pixel, y_pixel, start_time, end_time]\n\nTimes are in seconds.\nCoordinates are image pixels.",
    fontsize=12,
    va="top",
)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()

## 6. Compute saliency metrics

These metrics evaluate how well a saliency map predicts human fixations.

For a first sanity check, compare the human saliency map against the human fixation map itself.

In [ ]:
auc = ft.auc_judd(saliency_map, fixation_map, jitter=False)
nss = ft.nss(saliency_map, fixation_map)
kl = ft.kl_divergence(saliency_map, fixation_map)

print(f"AUC-Judd:      {auc:.4f}")
print(f"NSS:           {nss:.4f}")
print(f"KL-divergence: {kl:.4f}")

## 7. Compare two subjects with scanpath metrics

Now switch from saliency prediction to scanpath similarity.
We will compare the first two available subjects on the same image.

In [ ]:
subject_a, subject_b = subjects[:2]
scanpath_a = ft.scanpath(dataset_name, stimulus_name, subject=subject_a)
scanpath_b = ft.scanpath(dataset_name, stimulus_name, subject=subject_b)

scanmatch_score = ft.scanmatch_metric(
    scanpath_a,
    scanpath_b,
    stimulus_width=stimulus.shape[1],
    stimulus_height=stimulus.shape[0],
)
edit_distance, string_a, string_b = ft.string_edit_distance(
    scanpath_a[:, :2],
    scanpath_b[:, :2],
    stimulus_width=stimulus.shape[1],
    stimulus_height=stimulus.shape[0],
    grid_size=4,
)
scaled_tde = ft.scaled_time_delay_embedding_distance(scanpath_a[:, :2], scanpath_b[:, :2], stimulus)

print("Subject A:", subject_a)
print("Subject B:", subject_b)
print(f"ScanMatch similarity: {scanmatch_score:.4f}")
print(f"String edit distance: {edit_distance}")
print(f"Scaled TDE similarity: {scaled_tde:.4f}")
print("String A prefix:", string_a[:20])
print("String B prefix:", string_b[:20])

## 8. Inspect dataset metadata and integrity artifacts

This repository now ships machine-readable metadata, per-stimulus manifests, and checksums for each bundled dataset.
That makes it easier to validate the collection before running analyses.

In [ ]:
dataset_path = ft.COLLECTION_PATH / dataset_name
summary = dataset_utils.inspect_dataset(dataset_path)
validation = dataset_utils.validate_dataset_structure(dataset_path)

print("Dataset path:", dataset_path)
print("Summary keys:", sorted(summary.keys()))
print("Validation:")
for key, value in validation.items():
    print(f"  {key}: {value}")

## 9. Read the generated manifest and metadata files

These files are designed for reproducibility and tooling, not just human inspection.

In [ ]:
metadata_path = dataset_path / "metadata.json"
manifest_path = dataset_path / "manifest.csv"

print("metadata.json exists:", metadata_path.exists())
print("manifest.csv exists:", manifest_path.exists())
print("checksums.md5 exists:", (dataset_path / "checksums.md5").exists())

with open(metadata_path, "r", encoding="utf-8") as handle:
    metadata_preview = handle.read().splitlines()[:20]

print("\n".join(metadata_preview))

## 10. What to do next

You have now seen the complete core workflow:
- discover a dataset
- load stimulus-level data
- visualize maps and scanpaths
- compute saliency and scanpath metrics
- validate the dataset artifacts on disk

Recommended next steps:
1. Repeat this workflow on `TORONTO` or `MIT1003`.
2. Compare multiple subject pairs and collect the results in a table.
3. Use the CLI for integrity checks:
   - `python -m FixaTons validate Dataset/SIENA12`
   - `python -m FixaTons verify-checksums Dataset/SIENA12`
4. Read `docs/tutorials/README.md` for the suggested tutorial path.